# YOLOv8 fine-tuning round — Roboflow + own-labeled frames

**Goal:** fine-tune the previous round's weights (`best_<PREV>.pt`) into the
next round's weights (`best_<THIS>.pt`) on the union of (a) the public
Roboflow football-players-detection dataset and (b) the frames we just
labeled ourselves in Label Studio.

**To start a new round:** edit cell 2 below — bump `PREV_VERSION` and
`THIS_VERSION`. The rest of the notebook reads those values via f-strings
so nothing else changes.

**Why fine-tune, not train from scratch:** the previous round's weights
already know what players/refs/ball look like. We want to nudge the model
toward our newly labeled clips (stadium, kit, lighting) without
catastrophic forgetting.

## Runtime
`Runtime → Change runtime type → GPU (T4)`

## Expected uploads
1. `own_labels_split.zip` — produced locally by `scripts/prepare_own_labels_split.py`.
2. `best_<PREV>.pt` — your previous round's fine-tuned weights (e.g. `best_v2.pt`
   if this round produces `best_v3.pt`).


In [ ]:
# 1. Verify GPU + install deps
!nvidia-smi
!pip install -q ultralytics supervision roboflow

In [ ]:
# 2. Per-round parameters — bump these between training rounds.
PREV_VERSION = 'v2'      # weights you fine-tune FROM (e.g. 'v2' for round 3)
THIS_VERSION = 'v3'      # weights you'll produce (e.g. 'v3' for round 3)

PREV_WEIGHTS_FILE = f'best_{PREV_VERSION}.pt'   # filename you'll upload
THIS_WEIGHTS_FILE = f'best_{THIS_VERSION}.pt'   # filename you'll download
RUN_NAME = f'yolov8m-{THIS_VERSION}'             # under runs/football/

print(f'This round: {PREV_VERSION} -> {THIS_VERSION}')
print(f'  upload:   own_labels_split.zip + {PREV_WEIGHTS_FILE}')
print(f'  produce:  models/{THIS_WEIGHTS_FILE}')


In [ ]:
# 3. Upload own_labels_split.zip and your previous-round weights.
# (The previous-round filename is in PREV_WEIGHTS_FILE — e.g. 'best_v2.pt'.)
# Select BOTH files when the picker opens.
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))
assert PREV_WEIGHTS_FILE in uploaded, \
    f'Did not see {PREV_WEIGHTS_FILE} in upload — rename your weights file or fix PREV_VERSION.'


In [ ]:
# 4. Unzip our own labels.
# Handles three possible zip layouts:
#   (a) {images/train, images/val, labels/train, labels/val, classes.txt}
#       — what scripts/prepare_own_labels_split.py produces (already split)
#   (b) <wrapper>/{images/train, ...}  — hand-zipped from Explorer
#   (c) {images, labels, classes.txt} flat (no train/val) — when the zip was
#       made from datasets/own_labels/ instead of datasets/own_labels_split/.
#       In that case we do the deterministic 80/20 split in-place.
!mkdir -p /content/own && unzip -qo own_labels_split.zip -d /content/own

import shutil, hashlib
from pathlib import Path

own = Path('/content/own')

# --- Step 1: flatten any single wrapper directory (layout b) ------------
if not (own / 'images').exists():
    candidates = [p for p in own.iterdir() if p.is_dir() and (p / 'images').exists()]
    if len(candidates) == 1:
        wrapper = candidates[0]
        print(f'Flattening wrapper directory: {wrapper.name}/')
        for child in wrapper.iterdir():
            target = own / child.name
            if target.exists():
                shutil.rmtree(target) if target.is_dir() else target.unlink()
            shutil.move(str(child), str(target))
        wrapper.rmdir()
    else:
        raise RuntimeError(
            f'Cannot find images/ in /content/own. Top-level entries: '
            f'{sorted(p.name for p in own.iterdir())}'
        )

# --- Step 2: split flat layout into train/val if needed (layout c) ------
def _is_val(stem: str, val_frac: float = 0.2) -> bool:
    h = int(hashlib.md5(stem.encode('utf-8')).hexdigest(), 16)
    return (h % 1_000_000) / 1_000_000 < val_frac

if not (own / 'images' / 'train').exists():
    print('Flat images/labels detected — performing deterministic 80/20 split.')
    img_src = own / 'images'
    lbl_src = own / 'labels'
    flat_imgs = sorted(p for p in img_src.glob('*.jpg') if p.is_file())
    if not flat_imgs:
        raise RuntimeError(f'No .jpg files in {img_src}')
    for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
        (own / sub).mkdir(parents=True, exist_ok=True)
    n_train = n_val = n_skipped = 0
    for img in flat_imgs:
        lbl = lbl_src / f'{img.stem}.txt'
        if not lbl.exists():
            n_skipped += 1
            continue
        bucket = 'val' if _is_val(img.stem) else 'train'
        shutil.move(str(img), str(own / 'images' / bucket / img.name))
        shutil.move(str(lbl), str(own / 'labels' / bucket / lbl.name))
        if bucket == 'val':
            n_val += 1
        else:
            n_train += 1
    print(f'Split: {n_train} train | {n_val} val | {n_skipped} skipped (no label).')

!ls /content/own
!echo '---' && ls /content/own/images/train | head && echo '(...)' && ls /content/own/images/train | wc -l
!echo '---' && ls /content/own/images/val | wc -l


In [ ]:
# 5. Pull the Roboflow public dataset (same one we used last round).
# Swap the API key with yours from https://app.roboflow.com/settings/api
from roboflow import Roboflow
rf = Roboflow(api_key='PASTE_YOUR_ROBOFLOW_API_KEY')
project = rf.workspace('roboflow-jvuqo').project('football-players-detection-3zvbc')
version = project.version(20)   # same version as last round — keep it pinned
rf_dataset = version.download('yolov8')
RF_DIR = rf_dataset.location
print('Roboflow dataset at:', RF_DIR)
!cat {RF_DIR}/data.yaml


In [ ]:
# 6. Merge own-labeled frames INTO the Roboflow dataset directories.
# Roboflow's YOLO export has train/ valid/ test/ each with images/ and labels/.
# We copy our own frames in, renaming with a prefix to avoid any filename clash.
import shutil, os
from pathlib import Path

RF = Path(RF_DIR)
OWN = Path('/content/own')

def merge(split_own, split_rf):
    img_src = OWN / 'images' / split_own
    lbl_src = OWN / 'labels' / split_own
    img_dst = RF / split_rf / 'images'
    lbl_dst = RF / split_rf / 'labels'
    img_dst.mkdir(parents=True, exist_ok=True)
    lbl_dst.mkdir(parents=True, exist_ok=True)
    n = 0
    for img in img_src.glob('*.jpg'):
        lbl = lbl_src / f'{img.stem}.txt'
        if not lbl.exists():
            continue
        shutil.copy2(img, img_dst / f'own_{img.name}')
        shutil.copy2(lbl, lbl_dst / f'own_{lbl.name}')
        n += 1
    return n

n_train = merge('train', 'train')
# Roboflow's val folder is 'valid', not 'val'
n_val = merge('val', 'valid' if (RF / 'valid').exists() else 'val')
print(f'Merged: {n_train} own frames -> train, {n_val} own frames -> val')
!ls {RF}/train/images | wc -l
!ls {RF}/valid/images 2>/dev/null | wc -l || ls {RF}/val/images | wc -l


In [ ]:
# 7. Sanity-check class order consistency.
# Roboflow's data.yaml classes MUST match our own_labels classes.txt.
# Both should be: ['ball', 'goalkeeper', 'player', 'referee']  (indices 0-3).
import yaml
with open(f'{RF_DIR}/data.yaml') as f:
    data_cfg = yaml.safe_load(f)
print('Roboflow classes:', data_cfg.get('names'))
print('Own classes:', open('/content/own/classes.txt').read().strip().splitlines())
assert data_cfg['names'] == ['ball', 'goalkeeper', 'player', 'referee'], \
    'Class order mismatch! Fix before training.'


In [ ]:
# 8. Fine-tune from PREV_WEIGHTS_FILE (NOT from yolov8m.pt).
# Starting from our existing weights preserves what the model learned in
# the previous round and adapts it to our new clips faster than a cold start.
from ultralytics import YOLO

model = YOLO(f'/content/{PREV_WEIGHTS_FILE}')
results = model.train(
    data=f'{RF_DIR}/data.yaml',
    epochs=50,                # reasonable upper bound with patience=8 early-stop
    imgsz=1280,               # matters a lot for ball recall — keep
    batch=8,                  # T4 can handle 8 @ 1280; drop to 6 if OOM
    patience=8,
    lr0=0.001,                # lower LR than default — fine-tuning, not starting over
    mosaic=0.3,               # lighter aug than default (0.5) — preserve Roboflow structure
    project='runs/football',
    name=RUN_NAME,
    exist_ok=True,
)


In [ ]:
# 9. Evaluate on val set — compare with the previous round if you have those numbers handy.
metrics = model.val()
print(f'== {THIS_VERSION} metrics ==')
print('mAP50-95:', metrics.box.map)
print('mAP50   :', metrics.box.map50)
print('mAP75   :', metrics.box.map75)
# Per-class mAP — especially interested in 'ball' (usually the weakest).
for i, name in enumerate(model.names.values()):
    print(f'  {name:12s}  mAP50: {metrics.box.ap50[i]:.3f}  mAP50-95: {metrics.box.ap[i]:.3f}')


In [ ]:
# 10. Download the new weights. Drop into models/{THIS_WEIGHTS_FILE} in the project.
import subprocess
found = subprocess.check_output(['find', '/content', '-name', 'best.pt']).decode().splitlines()
print('candidates:', found)
# Pick the run dir for THIS round, ignoring the uploaded /content/<prev>.pt.
this_run = [f for f in found if RUN_NAME in f or 'runs/' in f]
best_pt = this_run[-1] if this_run else found[-1]
print('downloading:', best_pt, '->', THIS_WEIGHTS_FILE)

# Rename in-place so the downloaded file already has the version suffix.
import shutil
target = f'/content/{THIS_WEIGHTS_FILE}'
shutil.copy2(best_pt, target)

from google.colab import files
files.download(target)


## After download

1. Move the downloaded `best_<THIS>.pt` (e.g. `best_v3.pt`) into the project at
   `models/best_<THIS>.pt`. The previous round's weights stay in place — never
   overwrite them, you'll want to A/B test.
2. Also download `runs/football/yolov8m-<THIS>/results.png` and
   `confusion_matrix.png` from the Colab file browser — useful for comparing
   this round vs the previous one.
3. Re-run the pipeline pointed at the new weights:
   ```powershell
   python main.py --input video_clips/clip_7.mp4 `
                  --output output_videos/clip_7_v3.mp4 `
                  --model models/best_v3.pt --mode custom `
                  --max-frames 300 --imgsz 1280 `
                  --calibration calibrations/clip_7.json
   ```
   (Omit `--use-stub` — the tracker stub is model-specific; we want fresh
   detections from the new weights.)
4. **Next round:** edit cell 2 (`PREV_VERSION = 'v3'`, `THIS_VERSION = 'v4'`)
   and re-run the notebook with the new label batch.
